# Week 2, §4.2 — Party brands as informational shortcuts

Voters observe noisy signals $z_j = \theta_j + \eta_j$ of candidate
policies $\theta_j = \mu_P + \varepsilon_j$. They can use the party brand
$\mu_P$ as a Bayesian prior. We measure how much the brand adds to
classification accuracy across signal-noise regimes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2026)


## (a)--(b) Posterior

Bayesian update: $\theta_j \mid z_j, P \sim \mathcal{N}(\mu^*,
\sigma^*{}^2)$ with

$$\mu^* = \frac{\tau^{-2} z_j + \sigma^{-2} \mu_P}{\tau^{-2} +
\sigma^{-2}}, \quad \sigma^*{}^2 = \frac{1}{\tau^{-2} + \sigma^{-2}}.$$

Limits:

* $\tau \to \infty$: signal noise dominates; weight on brand $\to 1$.
* $\sigma \to 0$: parties homogeneous; weight on brand $\to 1$ (the brand
  *is* the candidate's policy).


## (c) Policy-estimation MSE across $\tau$

Voters' task is to estimate $\theta_j$. We compare two estimators:

* signal-only: $\hat{\theta}_j^S = z_j$, giving MSE $= \tau^2$.
* Bayesian shrinkage: $\hat{\theta}_j^B = (\tau^{-2} z_j + \sigma^{-2}
  \mu_P)/(\tau^{-2} + \sigma^{-2})$, giving MSE $= \tau^2\sigma^2/(\tau^2
  + \sigma^2)$.

The brand's MSE reduction is $\tau^4/(\tau^2 + \sigma^2)$, which is
increasing in $\tau$ and decreasing in $\sigma$.

**Note on classification.** Under symmetric priors and equal variances, the
optimal Bayesian classifier collapses to the midpoint rule $z \lessgtr 0.5$,
so the brand provides no *classification* value. Its informational role is in
*policy estimation* — that is what Snyder--Ting's argument is about.


In [ ]:
muA, muB = 0.3, 0.7
sigma = 0.05

def simulate_mse(tau, n=1000, sigma=sigma, muA=muA, muB=muB):
    thA = rng.normal(muA, sigma, n)
    thB = rng.normal(muB, sigma, n)
    zA = thA + rng.normal(0, tau, n)
    zB = thB + rng.normal(0, tau, n)
    z = np.concatenate([zA, zB])
    theta = np.concatenate([thA, thB])
    mu = np.concatenate([np.full(n, muA), np.full(n, muB)])
    hatS = z
    inv_tau2, inv_sigma2 = 1 / tau ** 2, 1 / sigma ** 2
    hatB = (inv_tau2 * z + inv_sigma2 * mu) / (inv_tau2 + inv_sigma2)
    return ((hatS - theta) ** 2).mean(), ((hatB - theta) ** 2).mean()

taus = np.array([0.025, 0.05, 0.1, 0.2, 0.3, 0.5])
mse_signal, mse_bayes, mse_theory = [], [], []
for t in taus:
    s, b = simulate_mse(t)
    theory = t ** 2 * sigma ** 2 / (t ** 2 + sigma ** 2)
    mse_signal.append(s); mse_bayes.append(b); mse_theory.append(theory)
    print(f'tau = {t:.3f}: '
          f'signal-only MSE = {s:.5f} (theory = {t**2:.5f}), '
          f'Bayes MSE = {b:.5f} (theory = {theory:.5f})')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(taus, mse_signal, 'o-', label=r'signal-only ($\hat\theta = z$)')
ax.plot(taus, mse_bayes, 's-', label='Bayesian shrinkage')
ax.plot(taus, taus ** 2, 'k:', alpha=0.5, label=r'theory: $\tau^2$')
ax.set(xlabel=r'$\tau$ (signal noise)', ylabel='MSE',
       title='The brand reduces estimation MSE; gap grows with $\\tau$')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## (d) Brand dilution and Lupu (2016)

We simulate the comparative-static across $\sigma$ (intra-party
heterogeneity) — Lupu's 'brand dilution'. As $\sigma$ rises, the
Bayesian shrinkage estimator approaches the signal-only estimator
(brand carries less weight) and its MSE rises toward $\tau^2$.


In [ ]:
sigmas = np.array([0.025, 0.05, 0.1, 0.2, 0.3])
tau = 0.1
mse_bayes_sigma = []
for s in sigmas:
    _, b = simulate_mse(tau, sigma=s)
    mse_bayes_sigma.append(b)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sigmas, mse_bayes_sigma, 'o-', color='C2', label='Bayes MSE')
ax.axhline(tau ** 2, color='k', linestyle=':', alpha=0.5,
           label=r'signal-only MSE ($\tau^2$)')
ax.set(xlabel=r'$\sigma$ (intra-party heterogeneity)',
       ylabel='MSE',
       title=r'Brand dilution erodes informational value (at fixed $\tau$)')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


**Lupu (2016) prediction.** Brand dilution corresponds to rising $\sigma$
(parties become coalitions of heterogeneous factions) or to drift in
$\mu_P$ over time. Empirical implications:

* Higher electoral volatility (voters lose their informational anchor).
* Increased reliance on personal candidate signals ($z_j$).
* Asymmetric vulnerability: parties whose original brand was sharp suffer
  more from drift than parties whose brand was always vague.
